In [1]:
import pandas as pd
import numpy as np

In [11]:
# Load your dataframe
df = pd.read_csv("deeploc_data_cleaned.csv")   # must contain columns: protein_id, sequence

# Define the amino acid alphabet (20 standard AAs)
AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")
aa_to_index = {aa: idx for idx, aa in enumerate(AMINO_ACIDS)}

# Add two special categories
SPECIAL_UNUSUAL = "UNUSUAL"   # for X, U, B, Z
SPECIAL_PADDING = "PADDING"   # for padding sequences 

# Build index mapping
aa_to_index = {aa: idx for idx, aa in enumerate(AMINO_ACIDS)}
aa_to_index[SPECIAL_UNUSUAL] = len(aa_to_index)      # index 20
aa_to_index[SPECIAL_PADDING] = len(aa_to_index)      # index 21

In [14]:
def one_hot_encode_sequence(seq):
    """
    Convert a protein sequence string into a one-hot encoded numpy array.
    Shape: (len(seq), 22) where 22 = 20 standard AAs + 2 special categories (unusual and padding).
    """
    one_hot = np.zeros((len(seq), len(aa_to_index)), dtype=np.float32)
    
    for i, aa in enumerate(seq):
        if aa in aa_to_index:
            one_hot[i, aa_to_index[aa]] = 1.0
        elif aa in ("X", "U", "B", "Z"):
            # Unusual amino acids
            one_hot[i, aa_to_index[SPECIAL_UNUSUAL]] = 1.0
        else:
            # Anything else unexpected → treat as padding
            one_hot[i, aa_to_index[SPECIAL_PADDING]] = 1.0
    return one_hot

In [15]:
# Apply encoding to each sequence
df["one_hot"] = df["sequence"].apply(one_hot_encode_sequence)

In [16]:
# Example: access the one-hot array for a specific protein
example_id = df.iloc[0]["protein_id"]
example_one_hot = df.iloc[0]["one_hot"]

print("Protein ID:", example_id)
print("One-hot shape:", example_one_hot.shape)
print("One-hot encoding for the first sequence:\n", example_one_hot)
print("One-hot encoding for the first sequence (as list):\n", example_one_hot.tolist())

Protein ID: Q9H400
One-hot shape: (295, 22)
One-hot encoding for the first sequence:
 [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
One-hot encoding for the first sequence (as list):
 [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.

In [20]:
df.to_csv('deeploc_data_onehot.csv', index=False)

In [19]:
def test_one_hot_encoding():
    # Test standard amino acids
    for aa in AMINO_ACIDS:
        encoded = one_hot_encode_sequence(aa)
        assert encoded.shape == (1, 22)
        assert encoded.sum() == 1.0
        assert encoded[0, aa_to_index[aa]] == 1.0

    # Test unusual amino acids
    for aa in ["X", "U", "B", "Z"]:
        encoded = one_hot_encode_sequence(aa)
        assert encoded.shape == (1, 22)
        assert encoded.sum() == 1.0
        assert encoded[0, aa_to_index[SPECIAL_UNUSUAL]] == 1.0

    # Test other characters (padding, weird chars)
    for aa in ["-", "*", "?", "1", "@"]:
        encoded = one_hot_encode_sequence(aa)
        assert encoded.shape == (1, 22)
        assert encoded.sum() == 1.0
        assert encoded[0, aa_to_index[SPECIAL_PADDING]] == 1.0

    # Test multi-character sequence
    seq = "ACXU-*"
    encoded = one_hot_encode_sequence(seq)
    assert encoded.shape == (len(seq), 22)

    # Check each position
    assert encoded[0, aa_to_index["A"]] == 1.0
    assert encoded[1, aa_to_index["C"]] == 1.0
    assert encoded[2, aa_to_index[SPECIAL_UNUSUAL]] == 1.0  # X
    assert encoded[3, aa_to_index[SPECIAL_UNUSUAL]] == 1.0  # U
    assert encoded[4, aa_to_index[SPECIAL_PADDING]] == 1.0    # -
    assert encoded[5, aa_to_index[SPECIAL_PADDING]] == 1.0    # *

    print("All one-hot encoding tests passed!")

In [18]:
test_one_hot_encoding()

Testing amino acid: A, encoded shape: (1, 22), sum: 1.0
Testing amino acid: C, encoded shape: (1, 22), sum: 1.0
Testing amino acid: D, encoded shape: (1, 22), sum: 1.0
Testing amino acid: E, encoded shape: (1, 22), sum: 1.0
Testing amino acid: F, encoded shape: (1, 22), sum: 1.0
Testing amino acid: G, encoded shape: (1, 22), sum: 1.0
Testing amino acid: H, encoded shape: (1, 22), sum: 1.0
Testing amino acid: I, encoded shape: (1, 22), sum: 1.0
Testing amino acid: K, encoded shape: (1, 22), sum: 1.0
Testing amino acid: L, encoded shape: (1, 22), sum: 1.0
Testing amino acid: M, encoded shape: (1, 22), sum: 1.0
Testing amino acid: N, encoded shape: (1, 22), sum: 1.0
Testing amino acid: P, encoded shape: (1, 22), sum: 1.0
Testing amino acid: Q, encoded shape: (1, 22), sum: 1.0
Testing amino acid: R, encoded shape: (1, 22), sum: 1.0
Testing amino acid: S, encoded shape: (1, 22), sum: 1.0
Testing amino acid: T, encoded shape: (1, 22), sum: 1.0
Testing amino acid: V, encoded shape: (1, 22), s